In [0]:
from pyspark.sql import functions as F

In [0]:
# 1. Carregar as fontes tratadas
df_orders = spark.table("bronze_olist_orders_dataset").withColumn(
    "order_purchase_timestamp", 
    F.to_timestamp("order_purchase_timestamp", "yyyy-MM-dd HH:mm:ss")
)
df_items = spark.table("bronze_olist_order_items_dataset")
df_customers = spark.table("bronze_olist_customers_dataset")
df_reviews = spark.table("bronze_olist_order_reviews_dataset")

In [0]:
# 2. Criar a variável alvo solicitada pelo negócio: 1 se review >= 4, senão 0
df_reviews_target = df_reviews.select(
    "order_id",
    F.col("review_score").cast("int").alias("review_score"),
    F.when(F.col("review_score").cast("int") >= 4, 1).otherwise(0).alias("target_high_review")
).dropDuplicates(["order_id"])

In [0]:
# 3. Agregar itens do pedido 
df_items_agg = df_items.groupBy("order_id").agg(
    F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight"),
    F.count("order_item_id").alias("item_count")
)

In [0]:
# 4. Consolidar a tabela Gold (Feature Store / Model Ready)
df_gold_model = df_orders.join(df_customers, "customer_id", "inner") \
                         .join(df_items_agg, "order_id", "inner") \
                         .join(df_reviews_target, "order_id", "inner") \
                         .select(
                             "order_id",
                             "customer_id",
                             "customer_state",
                             "customer_city",
                             "order_status",
                             "order_purchase_timestamp",
                             "total_price",
                             "total_freight",
                             "item_count",
                             "review_score",
                             "target_high_review"
                         )

In [0]:
# Visualizar o resultado pronto para o modelo preditivo
display(df_gold_model.limit(10))

In [0]:
# Gravar a tabela Delta Gold final
df_gold_model.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_olist_model_ready")

print("Tabela Gold para modelagem preditiva criada com sucesso!")